In [1]:
# --- Instalación ---
!pip install pyspark==3.5 -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# --- Task 1: Importar Librerías Requeridas ---

# Puedes también usar esta sección para suprimir advertencias:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# Importaciones principales de PySpark
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.pipeline import PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import os # Para verificar archivos (opcional)

# FindSpark simplifica el proceso en algunos entornos (como SNL)
# import findspark
# findspark.init() # Descomenta estas dos líneas si usas findspark

print("Librerías importadas.")

# --- Task 2: Crear una SparkSession ---

# Crea la SparkSession. appName es un nombre para tu aplicación Spark.
spark = SparkSession.builder.appName("AirfoilNoisePrediction").getOrCreate()

print("SparkSession creada exitosamente.")

Librerías importadas.
SparkSession creada exitosamente.


In [3]:
# --- Task 3 (Parte 1): Descargar el archivo de datos ---

# Asegúrate de usar ESTE dataset específico para el proyecto
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv

print("Descarga del archivo CSV solicitada.")

--2025-04-30 17:54:26--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60682 (59K) [text/csv]
Saving to: ‘NASA_airfoil_noise_raw.csv’

NASA_airfoil_noise_ 100%[===================>]  59.26K  --.-KB/s    in 0.003s  

2025-04-30 17:54:26 (21.2 MB/s) - ‘NASA_airfoil_noise_raw.csv’ saved [60682/60682]

Descarga del archivo CSV solicitada.


In [4]:
# --- Task 3 (Parte 2): Cargar el dataset en un DataFrame de Spark ---

csv_path = "NASA_airfoil_noise_raw.csv"

# Cargamos el CSV. Asumimos que tiene encabezado (header=True)
# y que queremos inferir los tipos de datos (inferSchema=True)
df = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"DataFrame cargado desde '{csv_path}'.")

DataFrame cargado desde 'NASA_airfoil_noise_raw.csv'.


In [5]:
# --- Task 4: Imprimir las 5 primeras filas del dataset ---

print("Mostrando las 5 primeras filas:")
df.show(5)

# Imprimir esquema para ver nombres y tipos
print("Esquema del DataFrame:")
df.printSchema()

Mostrando las 5 primeras filas:
+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows

Esquema del DataFrame:
root
 |-- Frequency: integer (nullable = true)
 |-- AngleOfAttack: double (nullable = true)


In [6]:
# --- Task 6: Print the total number of rows in the dataset ---
# your code goes here
rowcount1 = df.count()
print(f"Número total de filas iniciales: {rowcount1}")

Número total de filas iniciales: 1522


In [7]:
# --- Task 7: Drop all the duplicate rows from the dataset ---
# Count before dropping
# print(f"Filas antes de dropDuplicates: {df.count()}")
df = df.dropDuplicates()
# Count after dropping
# print(f"Filas después de dropDuplicates: {df.count()}")
print("Filas duplicadas eliminadas.")

Filas duplicadas eliminadas.


In [8]:
# --- Task 8: Print the total number of rows in the dataset ---
# your code goes here
rowcount2 = df.count()
print(f"Número total de filas tras eliminar duplicados: {rowcount2}")

Número total de filas tras eliminar duplicados: 1503


In [9]:
# --- Task 9: Drop all the rows that contain null values from the dataset ---
# Count before dropping nulls (opcional)
# print(f"Filas antes de dropna: {df.count()}")
df = df.dropna()
# Count after dropping nulls (opcional)
# print(f"Filas después de dropna: {df.count()}")
print("Filas con valores nulos eliminadas.")

Filas con valores nulos eliminadas.


In [10]:
# --- Task 10: Print the total number of rows in the dataset ---
# your code goes here
rowcount3 = df.count()
print(f"Número total de filas tras eliminar nulos: {rowcount3}")

Número total de filas tras eliminar nulos: 1499


In [11]:
# --- Task 11: Rename the column "SoundLevel" to "SoundLevelDecibels" ---
# your code goes here
# Nota: Verifica el nombre exacto de la columna con df.printSchema() si es necesario.
# Asumiendo que la columna se llama 'SoundLevel' según la tarea.
original_col_name = "SoundLevel" # Ajusta si el nombre real es diferente
new_col_name = "SoundLevelDecibels"

# Verifica si la columna original existe antes de renombrar
if original_col_name in df.columns:
    df = df.withColumnRenamed(original_col_name, new_col_name)
    print(f"Columna '{original_col_name}' renombrada a '{new_col_name}'.")
    # Verificar el schema (opcional)
    # df.printSchema()
else:
    # Si la columna ya tiene el nombre nuevo o uno diferente, informa.
    if new_col_name in df.columns:
         print(f"La columna ya se llama '{new_col_name}'. No se requiere renombrar.")
    else:
         print(f"Advertencia: No se encontró la columna '{original_col_name}' para renombrar. Columnas actuales: {df.columns}")
         # Si la columna objetivo se llama diferente (ej. por inferSchema del CSV), usa ese nombre
         # Ejemplo: Si se llama 'Scaled sound pressure level', usa:
         # original_col_name = 'Scaled sound pressure level' # Ajusta al nombre real
         # if original_col_name in df.columns:
         #    df = df.withColumnRenamed(original_col_name, new_col_name)
         #    print(f"Columna '{original_col_name}' renombrada a '{new_col_name}'.")
         # else:
         #    print(f"Error: No se encontró la columna objetivo con nombre '{original_col_name}'.")

Columna 'SoundLevel' renombrada a 'SoundLevelDecibels'.


In [12]:
# --- Task 12: Save the dataframe in parquet format ---
parquet_path = "NASA_airfoil_noise_cleaned.parquet"
# your code goes here
df.write.mode("overwrite").parquet(parquet_path)
print(f"DataFrame limpio guardado en formato Parquet en: '{parquet_path}'")

DataFrame limpio guardado en formato Parquet en: 'NASA_airfoil_noise_cleaned.parquet'


In [13]:
# --- Part 1 - Evaluation ---

print("Part 1 - Evaluation")

# Asegurarse de que las variables de conteo existan
print("Total rows = ", rowcount1 if 'rowcount1' in locals() else "Variable 'rowcount1' no definida")
print("Total rows after dropping duplicate rows = ", rowcount2 if 'rowcount2' in locals() else "Variable 'rowcount2' no definida")
print("Total rows after dropping duplicate rows and rows with null values = ", rowcount3 if 'rowcount3' in locals() else "Variable 'rowcount3' no definida")

# Verificar la última columna del DataFrame actual 'df'
if 'df' in locals() and hasattr(df, 'columns'):
    print("New column name = ", df.columns[-1]) # La última columna debería ser la renombrada/objetivo
else:
    print("DataFrame 'df' no está definido para verificar la columna.")

print(f"{parquet_path} exists :", os.path.isdir(parquet_path))

Part 1 - Evaluation
Total rows =  1522
Total rows after dropping duplicate rows =  1503
Total rows after dropping duplicate rows and rows with null values =  1499
New column name =  SoundLevelDecibels
NASA_airfoil_noise_cleaned.parquet exists : True


# --- Part 2 - Create a Machine Learning Pipeline ---

In [14]:
# --- Part 2 - Create a Machine Learning Pipeline ---
print("\n--- Iniciando Parte 2: Creación del Pipeline ---")

# --- Task 1: Load data from "NASA_airfoil_noise_cleaned.parquet" ---
# your code goes here
df = spark.read.parquet(parquet_path)
print(f"DataFrame cargado desde '{parquet_path}'.")


--- Iniciando Parte 2: Creación del Pipeline ---
DataFrame cargado desde 'NASA_airfoil_noise_cleaned.parquet'.


In [17]:
# --- Task 2: Print the total number of rows in the dataset ---
# your code goes here
rowcount4 = df.count()
print(f"Número total de filas en el DataFrame limpio: {rowcount4}")
#  Mostrar schema y datos
df.printSchema()
df.show(5)

Número total de filas en el DataFrame limpio: 1499
root
 |-- Frequency: integer (nullable = true)
 |-- AngleOfAttack: double (nullable = true)
 |-- ChordLength: double (nullable = true)
 |-- FreeStreamVelocity: double (nullable = true)
 |-- SuctionSideDisplacement: double (nullable = true)
 |-- SoundLevelDecibels: double (nullable = true)

+---------+-------------+-----------+------------------+-----------------------+------------------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevelDecibels|
+---------+-------------+-----------+------------------+-----------------------+------------------+
|     4000|          3.0|     0.3048|              31.7|             0.00529514|           115.608|
|     3150|          2.0|     0.2286|              31.7|             0.00372371|           121.527|
|     2000|          7.3|     0.2286|              31.7|              0.0132672|           115.309|
|     2000|          5.4|     0.1524|              71.3|  

In [18]:
# --- Task 3: Define the VectorAssembler pipeline stage ---
# Stage 1 - Assemble the input columns into "features"
# Use all columns except the target "SoundLevelDecibels"

# Primero, obtén la lista de columnas de características
target_col = "SoundLevelDecibels"
feature_cols = [col for col in df.columns if col != target_col]
print(f"Columnas de características para ensamblar: {feature_cols}")

# Define el ensamblador
# your code goes here
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
print("VectorAssembler definido.")

Columnas de características para ensamblar: ['Frequency', 'AngleOfAttack', 'ChordLength', 'FreeStreamVelocity', 'SuctionSideDisplacement']
VectorAssembler definido.


In [19]:
# --- Task 4: Define the StandardScaler pipeline stage ---
# Stage 2 - Scale "features" into "scaledFeatures"

# your code goes here
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
print("StandardScaler definido.")


StandardScaler definido.


In [20]:
# --- Task 5: Define the Model creation pipeline stage ---
# Stage 3 - Create a LinearRegression stage to predict "SoundLevelDecibels"

# your code goes here
# Usará "scaledFeatures" como entrada y predecirá la columna "SoundLevelDecibels"
lr = LinearRegression(featuresCol="scaledFeatures", labelCol=target_col)
print("LinearRegression definido.")

LinearRegression definido.


In [21]:
# --- Task 6: Build the pipeline ---
# Build a pipeline using the above three stages (assembler, scaler, lr)

# your code goes here
pipeline = Pipeline(stages=[assembler, scaler, lr])
print("Pipeline construido con 3 etapas.")

Pipeline construido con 3 etapas.


In [22]:
# --- Task 7: Split the data ---
# Split into training (70%) and testing (30%) sets, use seed=42

# your code goes here
(trainingData, testingData) = df.randomSplit([0.7, 0.3], seed=42)

print(f"Datos divididos: {trainingData.count()} filas para entrenamiento, {testingData.count()} filas para prueba.")

Datos divididos: 1101 filas para entrenamiento, 398 filas para prueba.


In [23]:
# --- Task 8: Fit the pipeline ---
# Fit the pipeline using the training data

# your code goes here
print("Entrenando el pipeline...")
pipelineModel = pipeline.fit(trainingData)
print("Pipeline entrenado exitosamente!")

Entrenando el pipeline...
Pipeline entrenado exitosamente!


# --- Part 3 - Evaluate the Model ---

In [24]:


print("\nPart 2 - Evaluation")
print("Total rows = ", rowcount4 if 'rowcount4' in locals() else "Variable 'rowcount4' no definida")

# Verificar etapas del pipeline
if 'pipeline' in locals() and hasattr(pipeline, 'getStages'):
    ps = [str(x).split("_")[0] for x in pipeline.getStages()]
    if len(ps) >= 3:
        print("Pipeline Stage 1 = ", ps[0]) # Debería ser VectorAssembler
        print("Pipeline Stage 2 = ", ps[1]) # Debería ser StandardScaler
        print("Pipeline Stage 3 = ", ps[2]) # Debería ser LinearRegression
    else:
        print("El pipeline no tiene las 3 etapas esperadas.")
else:
    print("Variable 'pipeline' no definida.")

# Verificar columna objetivo del modelo LR
if 'lr' in locals() and hasattr(lr, 'getLabelCol'):
    print("Label column = ", lr.getLabelCol())
else:
    print("Variable 'lr' (LinearRegression) no definida.")


Part 2 - Evaluation
Total rows =  1499
Pipeline Stage 1 =  VectorAssembler
Pipeline Stage 2 =  StandardScaler
Pipeline Stage 3 =  LinearRegression
Label column =  SoundLevelDecibels


In [25]:

print("\n--- Iniciando Parte 3: Evaluación del Modelo ---")

# --- Task 1: Predict using the model ---
# Make predictions on testing data

# your code goes here
predictions = pipelineModel.transform(testingData)

print("Predicciones realizadas sobre el conjunto de prueba.")
print("Mostrando 5 predicciones (Label vs Prediction):")
predictions.select(target_col, "prediction").show(5)


--- Iniciando Parte 3: Evaluación del Modelo ---
Predicciones realizadas sobre el conjunto de prueba.
Mostrando 5 predicciones (Label vs Prediction):
+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376775|
|            133.42|127.37968204568844|
|           119.146| 130.3407742507451|
|           116.074|131.11016975113546|
|           134.319|127.12627360125104|
+------------------+------------------+
only showing top 5 rows



In [26]:
# --- Task 2: Print the MSE ---
# your code goes here

# Crear el evaluador para MSE
evaluator_mse = RegressionEvaluator(predictionCol="prediction", labelCol=target_col, metricName="mse")
# Evaluar
mse = evaluator_mse.evaluate(predictions)

print(f"Mean Squared Error (MSE) = {mse}")

Mean Squared Error (MSE) = 24.997666255024154


In [27]:
# --- Task 3: Print the MAE ---
# your code goes here

# Crear el evaluador para MAE
evaluator_mae = RegressionEvaluator(predictionCol="prediction", labelCol=target_col, metricName="mae")
# Evaluar
mae = evaluator_mae.evaluate(predictions)

print(f"Mean Absolute Error (MAE) = {mae}")

Mean Absolute Error (MAE) = 3.9136790958811947


In [28]:
# --- Task 4: Print the R-Squared(R2) ---
# your code goes here

# Crear el evaluador para R2
evaluator_r2 = RegressionEvaluator(predictionCol="prediction", labelCol=target_col, metricName="r2")
# Evaluar
r2 = evaluator_r2.evaluate(predictions)

print(f"R-Squared (R2) = {r2}")

R-Squared (R2) = 0.49596884089746285


In [29]:
# --- Part 3 - Evaluation ---

print("\nPart 3 - Evaluation")

# Asegurarse de que las métricas existan
print("Mean Squared Error = ", round(mse, 2) if 'mse' in locals() else "Variable 'mse' no definida")
print("Mean Absolute Error = ", round(mae, 2) if 'mae' in locals() else "Variable 'mae' no definida")
print("R Squared = ", round(r2, 2) if 'r2' in locals() else "Variable 'r2' no definida")

# Obtener intercepto del modelo dentro del pipeline
if 'pipelineModel' in locals():
    # El modelo LR es la última etapa en este pipeline
    lrModel = pipelineModel.stages[-1]
    print("Intercept = ", round(lrModel.intercept, 2))
else:
    print("Variable 'pipelineModel' no definida.")



Part 3 - Evaluation
Mean Squared Error =  25.0
Mean Absolute Error =  3.91
R Squared =  0.5
Intercept =  132.88


# --- Part 4 - Persist the Model ---

In [30]:

print("\n--- Iniciando Parte 4: Persistencia del Modelo ---")

# --- Task 1: Save the model to the path "Final_Project" ---
model_path = "Final_Project_Airfoil_Model" # Dando un nombre más específico
# Save the pipeline model as "Final_Project" (usando model_path)
# your code goes here
pipelineModel.write().overwrite().save(model_path) # Usar overwrite para poder re-ejecutar

print(f"Modelo del Pipeline guardado en: '{model_path}'")


--- Iniciando Parte 4: Persistencia del Modelo ---
Modelo del Pipeline guardado en: 'Final_Project_Airfoil_Model'


In [31]:
# --- Task 2: Load the model from the path "Final_Project" ---
# Load the pipeline model you have created in the previous step
loadedPipelineModel = PipelineModel.load(model_path)
print(f"Modelo del Pipeline cargado desde: '{model_path}'")

Modelo del Pipeline cargado desde: 'Final_Project_Airfoil_Model'


In [32]:
# --- Task 3: Make predictions using the loaded model on the testdata ---
# Use the loaded pipeline model and make predictions using testingData
predictions_loaded = loadedPipelineModel.transform(testingData)
print("Predicciones realizadas con el modelo cargado.")

Predicciones realizadas con el modelo cargado.


In [33]:
# --- Task 4: Show the predictions ---
# show top 5 rows from the predictions dataframe. Display only the label column and predictions
# your code goes here
print("Mostrando 5 predicciones (Label vs Prediction) usando modelo cargado:")
predictions_loaded.select(target_col, "prediction").show(5)

Mostrando 5 predicciones (Label vs Prediction) usando modelo cargado:
+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376775|
|            133.42|127.37968204568844|
|           119.146| 130.3407742507451|
|           116.074|131.11016975113546|
|           134.319|127.12627360125104|
+------------------+------------------+
only showing top 5 rows



In [34]:
# --- Part 4 - Evaluation ---

print("\nPart 4 - Evaluation")

if 'loadedPipelineModel' in locals():
    # El modelo LR es la última etapa
    loadedmodel = loadedPipelineModel.stages[-1]
    totalstages = len(loadedPipelineModel.stages)
    # El VectorAssembler es la primera etapa (índice 0)
    inputcolumns = loadedPipelineModel.stages[0].getInputCols()

    print("Number of stages in the loaded pipeline = ", totalstages)
    print("Coefficients from loaded model:")
    # Asegurarse de que los coeficientes existan
    if hasattr(loadedmodel, 'coefficients'):
        for i, j in zip(inputcolumns, loadedmodel.coefficients):
            print(f"  Coefficient for {i} is {round(j, 4)}")
    else:
        print("El modelo cargado no tiene coeficientes accesibles.")
else:
    print("Variable 'loadedPipelineModel' no definida.")


Part 4 - Evaluation
Number of stages in the loaded pipeline =  3
Coefficients from loaded model:
  Coefficient for Frequency is -3.9906
  Coefficient for AngleOfAttack is -2.2881
  Coefficient for ChordLength is -3.3269
  Coefficient for FreeStreamVelocity is 1.4832
  Coefficient for SuctionSideDisplacement is -2.0551


In [35]:
# --- Stop Spark Session ---
print("\nDeteniendo SparkSession...")
spark.stop()
print("SparkSession detenida.")


Deteniendo SparkSession...
SparkSession detenida.
